# 06 - Transitive Graph-Diff Fix: Catching Multi-Hop Downstream Changes

Companion notebook to `../07-production-resilience-and-operational-engineering.md`, bug narrative
#3: *"A graph-diff bug in Country Specific Protocol Comparison missed a downstream requirement
change because the traversal only checked direct edges, not transitive ones."*

This notebook extends `03_graphdb_cypher_queries_demo.ipynb`'s `diff_for_country` function --
reusing the exact same requirement graph and `REFERENCES` edge model -- with a **two-hop reference
chain** (`REQ-EXCL-01` references `REQ-ELIG-02`, which references `REQ-LAB-01`) that the original
one-hop implementation cannot see past. The buggy, one-hop version and the fixed, transitive version
are built and run side by side against the identical graph, so the missed change is demonstrated
concretely, not just asserted.

Fully offline. Standard library only (an optional `networkx` section at the end mirrors notebook
03's pattern, and is skipped gracefully if `networkx` isn't installed).

## 1. The same requirement graph as notebook 03, extended with a two-hop reference chain

Notebook 03's graph has exactly one `REFERENCES` edge: `REQ-ELIG-02 -> REQ-LAB-01` (an eligibility
criterion that references a lab reference-range requirement). This notebook adds one more
requirement, `REQ-EXCL-01` (an exclusion criterion), that references `REQ-ELIG-02` -- producing the
chain chapter 07's bug narrative describes: **A references B, and B references C.** If Germany
modifies `REQ-LAB-01` (= C), a one-hop traversal correctly finds `REQ-ELIG-02` (= B, which directly
references C) but misses `REQ-EXCL-01` (= A) entirely, even though A's meaning depends on C
transitively through B.

In [1]:
GLOBAL_REQUIREMENTS = {
    "REQ-ELIG-01": {
        "type": "eligibility",
        "text": "Adults aged 18 to 75 years, inclusive, at the time of screening.",
    },
    "REQ-ELIG-02": {
        "type": "eligibility",
        "text": "Pre-bronchodilator FEV1 of 40-90% of predicted normal value at screening, per REQ-LAB-01.",
    },
    "REQ-LAB-01": {
        "type": "lab",
        "text": "Spirometry reference ranges per standard adult population norms.",
    },
    "REQ-DOSE-01": {
        "type": "dosing",
        "text": "100 mg subcutaneously every 4 weeks for participants weighing less than 60 kg.",
    },
    "REQ-SAFETY-01": {
        "type": "safety",
        "text": "Monitor for injection site reactions for 30 minutes following each dose.",
    },
    # NEW: an exclusion criterion that references the eligibility criterion above -- the second
    # hop in the chain this notebook is built around. A REFERENCES B REFERENCES C.
    "REQ-EXCL-01": {
        "type": "eligibility",
        "text": (
            "Participants who do not meet the pulmonary function criteria described in "
            "REQ-ELIG-02 are excluded from randomization."
        ),
    },
}

# REQ-ELIG-02 (B) depends on REQ-LAB-01 (C); REQ-EXCL-01 (A) depends on REQ-ELIG-02 (B).
# A -> B -> C: a two-hop chain, exactly chapter 07 bug #3's scenario.
REFERENCES = [
    ("REQ-ELIG-02", "REQ-LAB-01"),   # B -> C (the one hop the buggy traversal already finds)
    ("REQ-EXCL-01", "REQ-ELIG-02"),  # A -> B (the second hop the buggy traversal misses)
]

print(f"{len(GLOBAL_REQUIREMENTS)} requirement nodes, {len(REFERENCES)} REFERENCES edges")
print("Chain: REQ-EXCL-01 -> REQ-ELIG-02 -> REQ-LAB-01")

6 requirement nodes, 2 REFERENCES edges
Chain: REQ-EXCL-01 -> REQ-ELIG-02 -> REQ-LAB-01


## 2. Germany modifies the lab requirement -- same variant setup as notebook 03

In [2]:
COUNTRY_VARIANTS = {
    "Germany": {
        "MODIFIES": {
            "REQ-LAB-01": (
                "Spirometry reference ranges per German national pulmonology society (DGP) "
                "norms, not generic adult norms."
            ),
        },
        "ADDS": {},
    },
}

def build_graph():
    nodes = {"requirement": dict(GLOBAL_REQUIREMENTS), "country_variant": {}}
    edges = {"REFERENCES": list(REFERENCES), "MODIFIES": [], "ADDS": []}

    for country, changes in COUNTRY_VARIANTS.items():
        variant_id = f"VARIANT-{country.upper()}"
        nodes["country_variant"][variant_id] = {"country": country}
        for req_id, new_text in changes.get("MODIFIES", {}).items():
            edges["MODIFIES"].append((variant_id, req_id, new_text))
        for req_id, req_data in changes.get("ADDS", {}).items():
            nodes["requirement"][req_id] = req_data
            edges["ADDS"].append((variant_id, req_id))
    return nodes, edges


nodes, edges = build_graph()
print("MODIFIES edges:", edges["MODIFIES"])
print("REFERENCES edges:", edges["REFERENCES"])

MODIFIES edges: [('VARIANT-GERMANY', 'REQ-LAB-01', 'Spirometry reference ranges per German national pulmonology society (DGP) norms, not generic adult norms.')]
REFERENCES edges: [('REQ-ELIG-02', 'REQ-LAB-01'), ('REQ-EXCL-01', 'REQ-ELIG-02')]


## 3. The bug, reproduced: `diff_for_country_one_hop` (identical to notebook 03's `diff_for_country`)

This is the exact traversal from `03_graphdb_cypher_queries_demo.ipynb` -- unchanged, reproduced here
to demonstrate the gap concretely. It checks `dst in changed_ids`, which only ever catches
requirements **directly** connected to something that changed -- one hop, no further.

In [3]:
def diff_for_country_one_hop(nodes, edges, country: str):
    """The pre-fix traversal (chapter 07 bug #3): only checks direct REFERENCES edges into a
    changed requirement. Correctly finds REQ-ELIG-02 (one hop from REQ-LAB-01) but has no
    mechanism to look past that first hop."""
    variant_id = f"VARIANT-{country.upper()}"

    modified = [(req_id, new_text) for (vid, req_id, new_text) in edges["MODIFIES"] if vid == variant_id]
    added = [req_id for (vid, req_id) in edges["ADDS"] if vid == variant_id]

    changed_ids = {req_id for req_id, _ in modified} | set(added)
    downstream = [
        (src, dst) for (src, dst) in edges["REFERENCES"]
        if dst in changed_ids and src not in changed_ids
    ]

    return {"modified": modified, "added": added, "downstream_affected": downstream}


buggy_diff = diff_for_country_one_hop(nodes, edges, "Germany")
print("One-hop diff for Germany:")
print("  modified:            ", buggy_diff["modified"])
print("  downstream_affected: ", buggy_diff["downstream_affected"])

downstream_ids = {src for src, _ in buggy_diff["downstream_affected"]}
print("\nRequirements the one-hop traversal flags as downstream-affected:", downstream_ids)

# The bug, demonstrated directly: REQ-EXCL-01 depends on REQ-LAB-01 transitively (through
# REQ-ELIG-02), but the one-hop traversal has no way to see it.
assert "REQ-ELIG-02" in downstream_ids, "the direct, one-hop dependent must still be found"
assert "REQ-EXCL-01" not in downstream_ids, \
    "this is the bug: the one-hop traversal misses the second-hop dependent entirely"
print("\nCONFIRMED BUG: REQ-EXCL-01 depends on REQ-LAB-01 (through REQ-ELIG-02), but the one-hop")
print("traversal never surfaces it -- exactly the near-miss chapter 07 bug #3 describes, caught by")
print("a reviewer manually reading the section, not by the pipeline.")

One-hop diff for Germany:
  modified:             [('REQ-LAB-01', 'Spirometry reference ranges per German national pulmonology society (DGP) norms, not generic adult norms.')]
  downstream_affected:  [('REQ-ELIG-02', 'REQ-LAB-01')]

Requirements the one-hop traversal flags as downstream-affected: {'REQ-ELIG-02'}

CONFIRMED BUG: REQ-EXCL-01 depends on REQ-LAB-01 (through REQ-ELIG-02), but the one-hop
traversal never surfaces it -- exactly the near-miss chapter 07 bug #3 describes, caught by
a reviewer manually reading the section, not by the pipeline.


## 4. The fix: `diff_for_country_transitive` -- follow `REFERENCES` edges to a fixed point

Instead of checking one hop, walk the **reverse-reference graph** (build an index of "who
references me" for every requirement) and run a breadth-first search outward from every directly
changed requirement, following edges however many hops it takes until no new dependents are found.
Every requirement discovered this way -- regardless of hop distance -- is downstream-affected.

In [4]:
from collections import deque


def build_reverse_reference_index(edges):
    """dst -> [src, src, ...] -- for a given requirement, which requirements REFERENCE it directly.
    This is the index the BFS below walks outward from a changed requirement."""
    reverse_index: dict[str, list[str]] = {}
    for src, dst in edges["REFERENCES"]:
        reverse_index.setdefault(dst, []).append(src)
    return reverse_index


def diff_for_country_transitive(nodes, edges, country: str):
    """The fixed traversal: breadth-first search outward from every directly changed requirement,
    following REFERENCES edges however many hops it takes, until no new dependents are found.
    Returns every downstream-affected requirement together with its hop distance from the nearest
    directly changed requirement, so a reviewer can see not just THAT something is affected but
    HOW -- via which chain of references."""
    variant_id = f"VARIANT-{country.upper()}"

    modified = [(req_id, new_text) for (vid, req_id, new_text) in edges["MODIFIES"] if vid == variant_id]
    added = [req_id for (vid, req_id) in edges["ADDS"] if vid == variant_id]
    changed_ids = {req_id for req_id, _ in modified} | set(added)

    reverse_index = build_reverse_reference_index(edges)

    downstream_affected: dict[str, dict] = {}   # req_id -> {"hops": int, "path": [...]}
    queue = deque((req_id, [req_id], 1) for req_id in changed_ids)
    visited = set(changed_ids)

    while queue:
        current, path, hops = queue.popleft()
        for dependent in reverse_index.get(current, []):
            if dependent in visited:
                continue
            visited.add(dependent)
            dependent_path = [dependent] + path
            downstream_affected[dependent] = {"hops": hops, "path": dependent_path}
            queue.append((dependent, dependent_path, hops + 1))

    return {
        "modified": modified,
        "added": added,
        "downstream_affected": downstream_affected,
    }


fixed_diff = diff_for_country_transitive(nodes, edges, "Germany")
print("Transitive diff for Germany:")
print("  modified:", fixed_diff["modified"])
print("  downstream_affected:")
for req_id, info in fixed_diff["downstream_affected"].items():
    path_str = " -> ".join(info["path"])
    print(f"    {req_id}  ({info['hops']} hop{'s' if info['hops'] != 1 else ''} away)  path: {path_str}")

Transitive diff for Germany:
  modified: [('REQ-LAB-01', 'Spirometry reference ranges per German national pulmonology society (DGP) norms, not generic adult norms.')]
  downstream_affected:
    REQ-ELIG-02  (1 hop away)  path: REQ-ELIG-02 -> REQ-LAB-01
    REQ-EXCL-01  (2 hops away)  path: REQ-EXCL-01 -> REQ-ELIG-02 -> REQ-LAB-01


## 5. The fix, verified: the second-hop dependent is now found, with its full path

Both `REQ-ELIG-02` (one hop) and `REQ-EXCL-01` (two hops) must now appear, and `REQ-EXCL-01`'s
recorded path should show it depends on the changed requirement *through* `REQ-ELIG-02` --
information a reviewer can use to actually understand why it's flagged, not just that it is.

In [5]:
downstream_ids_fixed = set(fixed_diff["downstream_affected"].keys())
print("Requirements flagged as downstream-affected by the FIXED traversal:", downstream_ids_fixed)

assert "REQ-ELIG-02" in downstream_ids_fixed
assert fixed_diff["downstream_affected"]["REQ-ELIG-02"]["hops"] == 1

assert "REQ-EXCL-01" in downstream_ids_fixed, "the fix must catch the second-hop dependent"
assert fixed_diff["downstream_affected"]["REQ-EXCL-01"]["hops"] == 2
assert fixed_diff["downstream_affected"]["REQ-EXCL-01"]["path"] == ["REQ-EXCL-01", "REQ-ELIG-02", "REQ-LAB-01"]

print("\nFIX CONFIRMED: REQ-EXCL-01 is now correctly flagged as downstream-affected, two hops away,")
print("with its full dependency chain recorded (REQ-EXCL-01 -> REQ-ELIG-02 -> REQ-LAB-01) -- the")
print("exact gap the one-hop traversal in Section 3 left silently unflagged.")

Requirements flagged as downstream-affected by the FIXED traversal: {'REQ-EXCL-01', 'REQ-ELIG-02'}

FIX CONFIRMED: REQ-EXCL-01 is now correctly flagged as downstream-affected, two hops away,
with its full dependency chain recorded (REQ-EXCL-01 -> REQ-ELIG-02 -> REQ-LAB-01) -- the
exact gap the one-hop traversal in Section 3 left silently unflagged.


## 6. Side by side: the buggy and fixed results on the identical graph

The point worth making explicit: nothing about the *graph* or the *change* differs between Sections
3 and 4 -- only the traversal logic does. This is precisely why chapter 07 says a one-hop-only test
fixture set can never catch this bug: it would pass against both implementations identically, because
a one-hop fixture set has no case that distinguishes "stops after one hop" from "correctly follows the
chain to a fixed point." 

In [6]:
print(f"{'Requirement':15s} {'One-hop traversal':20s} {'Transitive traversal'}")
print("-" * 60)
for req_id in ["REQ-ELIG-02", "REQ-EXCL-01"]:
    one_hop_found = req_id in {src for src, _ in buggy_diff["downstream_affected"]}
    transitive_found = req_id in downstream_ids_fixed
    print(f"{req_id:15s} {'FOUND' if one_hop_found else 'MISSED':20s} {'FOUND' if transitive_found else 'MISSED'}")

assert not ({"REQ-ELIG-02", "REQ-EXCL-01"} <= {src for src, _ in buggy_diff["downstream_affected"]})
assert {"REQ-ELIG-02", "REQ-EXCL-01"} <= downstream_ids_fixed
print("\nConfirmed: the transitive traversal is a strict superset of the one-hop traversal's")
print("findings on this graph -- it never loses a direct match, and it additionally catches the")
print("chained, multi-hop dependent the original implementation silently missed.")

Requirement     One-hop traversal    Transitive traversal
------------------------------------------------------------
REQ-ELIG-02     FOUND                FOUND
REQ-EXCL-01     MISSED               FOUND

Confirmed: the transitive traversal is a strict superset of the one-hop traversal's
findings on this graph -- it never loses a direct match, and it additionally catches the
chained, multi-hop dependent the original implementation silently missed.


## 7. Optional: the same fixed traversal expressed as a variable-length Cypher path

Matching notebook 03's pattern of showing the equivalent Cypher alongside the Python. The one-hop
buggy query and the fixed, variable-length-path query differ by exactly one syntactic detail --
`[:REFERENCES*1..]` instead of `[:REFERENCES]` -- which is itself a useful thing to be able to say in
an interview: *"the fix isn't a rewrite, it's changing a fixed-length pattern to a variable-length
one."*

```cypher
-- BUGGY (one-hop only) -- matches notebook 03's Section 3 query exactly:
MATCH (c:CountryVariant {country: "Germany"})-[:MODIFIES]->(changed:Requirement)
MATCH (dependent:Requirement)-[:REFERENCES]->(changed)
RETURN dependent.id, dependent.text, changed.id AS depends_on_changed_requirement

-- FIXED (variable-length path, one or more hops):
MATCH (c:CountryVariant {country: "Germany"})-[:MODIFIES]->(changed:Requirement)
MATCH path = (dependent:Requirement)-[:REFERENCES*1..]->(changed)
RETURN dependent.id, dependent.text, length(path) AS hops, changed.id AS depends_on_changed_requirement
```

In [7]:
try:
    import networkx as nx

    G = nx.DiGraph()
    for req_id in nodes["requirement"]:
        G.add_node(req_id)
    for src, dst in edges["REFERENCES"]:
        G.add_edge(src, dst)

    # networkx's built-in ancestors() is exactly "everything that can reach this node" -- i.e.
    # everything that transitively REFERENCES it, directly or through a chain. A convenient,
    # library-provided sanity check against the hand-rolled BFS above.
    changed_ids = {"REQ-LAB-01"}
    nx_downstream = set()
    for changed in changed_ids:
        nx_downstream |= nx.ancestors(G, changed)

    print("networkx nx.ancestors() downstream set:", nx_downstream)
    assert nx_downstream == downstream_ids_fixed
    print("OK: networkx's ancestors() traversal agrees exactly with the hand-rolled transitive BFS.")

except ImportError:
    print(
        "networkx is not installed in this environment -- that's fine, nothing above depends on "
        "it. The hand-rolled BFS in Section 4 is the complete, fully offline transitive-diff "
        "implementation; networkx's nx.ancestors() (or Cypher's variable-length path syntax, "
        "Section 7) would simply be a more concise way to express the identical traversal."
    )

networkx nx.ancestors() downstream set:

 {'REQ-EXCL-01', 'REQ-ELIG-02'}
OK: networkx's ancestors() traversal agrees exactly with the hand-rolled transitive BFS.


## Takeaways

- **The bug is a traversal depth, not a modeling error.** The graph already had everything needed to
  find `REQ-EXCL-01` -- the `REFERENCES` edges were there, correctly modeled. The one-hop function
  just never looked past the first edge (Section 3).
- **A fixed point, not a fixed hop count, is the correct stopping condition.** `diff_for_country_transitive`
  keeps expanding via BFS until no new dependents are discovered, so it correctly handles chains of
  any length -- two hops here, but the same function handles three, four, or more without modification
  (Section 4).
- **Recording the path, not just the fact of being affected, matters for reviewer trust.** `REQ-EXCL-01`'s
  entry shows *why* it's flagged (`REQ-EXCL-01 -> REQ-ELIG-02 -> REQ-LAB-01`) -- a bare "affected: yes"
  flag with no explanation is a harder thing for a regulatory-affairs reviewer to act on confidently
  (Section 5).
- **A one-hop-only test suite cannot distinguish correct from buggy.** Section 6 makes this concrete:
  the two traversals agree completely except on the one case (`REQ-EXCL-01`) that specifically requires
  a multi-hop chain to exercise -- exactly why chapter 07 calls out "test cases that specifically cover
  chains of two or more `REFERENCES` edges" as the fix that would have caught this earlier, not just a
  bigger pile of one-hop fixtures.